In [ ]:
!pip -q install scipy scikit-learn spectral torchinfo

import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import spectral

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary

Loading Dataset As an NP array

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np

DATA_PATH = '/content/drive/MyDrive/Data/Indian Pines HSI/'

data   = np.load(DATA_PATH + 'indianpinearray.npy')  # hyperspectral image
labels = np.load(DATA_PATH + 'IPgt.npy')             # ground truth

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

PCA Application

In [ ]:
def apply_pca(X, num_components=30):
    newX = np.reshape(X, (-1, X.shape[2]))
    pca  = PCA(n_components=num_components, whiten=True)
    newX = pca.fit_transform(newX)
    return np.reshape(newX, (X.shape[0], X.shape[1], num_components)), pca

data_pca, pca = apply_pca(data, num_components=30)

Create Patches

In [ ]:
def create_patches(X, y, patch_size):
    margin = patch_size // 2
    padded_X = np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')
    patches_list, labels_list = [], []

    for i in range(margin, padded_X.shape[0] - margin):
        for j in range(margin, padded_X.shape[1] - margin):
            patch = padded_X[i - margin:i + margin + 1, j - margin:j + margin + 1, :]
            patches_list.append(patch)
            labels_list.append(y[i - margin, j - margin])

    return np.array(patches_list), np.array(labels_list)

PATCH_SIZE = 15
patches, patch_labels = create_patches(data_pca, labels, PATCH_SIZE)

Data Split

In [ ]:
non_zero = patch_labels > 0
patches = patches[non_zero]
patch_labels = patch_labels[non_zero] - 1

X_train, X_test, y_train, y_test = train_test_split(
    patches, patch_labels, test_size=0.3, random_state=42
)

X_train = np.transpose(X_train, (0, 3, 1, 2)).astype(np.float32)
X_test  = np.transpose(X_test,  (0, 3, 1, 2)).astype(np.float32)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64)

num_classes = int(np.unique(patch_labels).shape[0])
in_channels = data_pca.shape[-1]

Model

In [ ]:
import torch.nn.functional as F

class SelfAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (B, N, C)
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + attn_out)


class CNN_Attention_Model(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()

        # CNN feature extractor
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 15 → 7

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 7 → 3
        )

        # Self-attention
        self.attention = SelfAttentionBlock(embed_dim=128, num_heads=4)

        # Classification head
        self.fc = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)   # (B, 128, 3, 3)

        B, C, H, W = x.shape

        # Flatten spatial → sequence
        x = x.view(B, C, H*W).permute(0, 2, 1)   # (B, 9, 128)

        # Apply self-attention
        x = self.attention(x)

        # Global average pooling over sequence
        x = x.mean(dim=1)   # (B, 128)

        return self.fc(x)


model = CNN_Attention_Model(in_channels, num_classes).to(device)

Training Setup


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20

Main Training

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Accuracy Evaluation

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        outputs = model(xb)
        preds = outputs.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

accuracy = correct / total
print("Test Accuracy:", accuracy)

Visualization of the results

In [ ]:
def visualize_results(data_pca, labels, model, patch_size):
    model.eval()
    margin = patch_size // 2
    padded = np.pad(data_pca, ((margin, margin), (margin, margin), (0, 0)), mode='constant')

    H, W, _ = data_pca.shape
    preds = np.zeros((H, W))

    with torch.no_grad():
        for i in range(H):
            for j in range(W):
                patch = padded[i:i+patch_size, j:j+patch_size]
                patch = torch.from_numpy(patch.transpose(2,0,1)).unsqueeze(0).float().to(device)
                pred = model(patch).argmax().item()
                preds[i, j] = pred

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.title("Ground Truth"); plt.imshow(labels)
    plt.subplot(1,2,2); plt.title("Prediction"); plt.imshow(preds)
    plt.show()

visualize_results(data_pca, labels, model, PATCH_SIZE)